<a href="https://colab.research.google.com/github/Farahani1/kodum/blob/claude%2Fsweet-franklin-pm0tdm/notebooks/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# kodoom on Colab

This notebook only connects Colab to the project: it mounts Drive, gets the code, installs it and runs `kodoom` commands. All logic lives in the repository, so what runs here is exactly what already ran on the laptop (`dev` profile).

**Every session:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

**After a session dies:** reconnect and Run all again. Logs and checkpoints are already on Drive; a run started again with the same run id resumes from its last checkpoint.

**To update the code:** push to the branch from the laptop, then re-run the *Get the code* cell (or Run all). See `docs/colab.md`.

## Settings

In [1]:
REF = "claude/sweet-franklin-pm0tdm"  # branch, tag or commit to run
PROFILE = "colab-preflight"  # colab-preflight first, then colab

## Mount Google Drive
Runs, logs and checkpoints are written to `MyDrive/kodoom/runs`. Nothing else goes to Drive.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


## Get the code
Clones the repository the first time and moves to the latest `REF` every time it runs, discarding any edits made on Colab: code changes are made on the laptop and pushed.

For a private repository, add a GitHub token under 🔑 Secrets (left sidebar) named `GITHUB_TOKEN`, with notebook access on. The token is used for each fetch and is not saved in the clone.

In [4]:
import os
import subprocess

REPO = "github.com/Farahani1/kodum.git"
DIR = "/content/kodoom"

try:
    from google.colab import userdata

    token = userdata.get("github-kodoom")
except Exception:  # no secret set: a public repository needs none
    token = None
url = f"https://x-access-token:{token}@{REPO}" if token else f"https://{REPO}"


def git(*args):
    result = subprocess.run(["git", *args], capture_output=True, text=True)
    if result.returncode != 0:
        message = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(message)  # git's own message, never the token
    return result.stdout.strip()


if not os.path.isdir(f"{DIR}/.git"):
    git("init", "--quiet", DIR)
git("-C", DIR, "fetch", "--quiet", "--depth", "50", url, REF)
git("-C", DIR, "checkout", "--quiet", "--force", "FETCH_HEAD")
print(git("-C", DIR, "log", "-1", "--format=%h %s (%cr)"))

6f37993 colab: say why the token was not used when a private fetch fails (56 seconds ago)


In [5]:
!pip install --quiet -e "{DIR}[colab]"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for kodoom (pyproject.toml) ... done


## Check the environment
Stops here if Drive is not mounted, no GPU is attached or Drive is too full. Also lists the runs that can be resumed.

In [6]:
!kodoom check --profile {PROFILE}

profile: colab-preflight
  [  ok] kodoom: 0.1.0, commit 6f37993c0ac72a21dead2345cc1b187352243842
  [  ok] python: 3.13.15 on Linux
  [  ok] runs_dir: /content/drive/MyDrive/kodoom/runs/preflight
  [  ok] free space: 14.7 GB free under /content/drive/MyDrive/kodoom/runs/preflight
  [  ok] resumable runs: none
  [  ok] scratch_dir: /content/kodoom-scratch
  [  ok] model cache: HF_HOME=/content/hf-cache
  [  ok] device: Tesla T4, 15 GB
ready


In [7]:
!kodoom runs --profile {PROFILE}

no runs under /content/drive/MyDrive/kodoom/runs/preflight


## Run
Pipeline steps are added here as they are built, each as one `kodoom` command with `--profile {PROFILE}`. Re-running a step with the same run id resumes it.

In [8]:
!kodoom info --profile {PROFILE}

kodoom 0.1.0
                  name: colab-preflight
                  seed: 1234
              runs_dir: /content/drive/MyDrive/kodoom/runs/preflight
                device: cuda
             precision: fp16
               threads: None
  max_cases_per_source: 200
           scratch_dir: /content/kodoom-scratch
             cache_dir: /content/hf-cache
            reserve_gb: 1.0


## Before closing (optional)
Everything is already on Drive as it is written. This forces Drive to finish syncing, for peace of mind before closing the tab.

In [ ]:
drive.flush_and_unmount()